# Step 0 — Know your data before you measure anything

**Purpose:** understand what VQA-RAD and SLAKE actually contain, *before* you
run a single GPU-hour. Four numbers here change decisions you make later.

| What you look at | Why it changes a decision |
|---|---|
| **Majority-class baseline** | If 60% of closed answers are `yes`, a model that always says `yes` scores 60%. That, not zero, is the floor your fine-tuned model must clear. |
| **Answer concentration** | If 15 answers cover most open questions, the task is closer to classification than generation, and open-ended recall flatters the model. |
| **Answer length** | Gold answers are 1–2 words. This is the evidence that justifies the constrained prompt. |
| **Image appearance** | Grayscale radiographs look nothing like the natural photos the vision encoder was pretrained on. That gap is the entire premise of the vision-side adaptation hypothesis. |

**No GPU needed.** `Runtime > Change runtime type > CPU` is fine and starts faster.

All plotting lives in `src/explore.py`, so every figure regenerates from one
command on a clean checkout. This notebook only calls it and shows the output.


## 1. Get the code


In [ ]:
# --- OPTION A: clone your repo (use this once pushed to GitHub) ---
# %cd /content
# !git clone https://github.com/YOUR_USERNAME/medvlm-module-ablation.git medvlm
# %cd /content/medvlm

# --- OPTION B: copy from Drive (fine for the very first run) ---
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/medvlm/code /content/medvlm 2>/dev/null || echo 'adjust this path'
%cd /content/medvlm
!ls -la


## 2. Install and cache

`HF_HOME` points at Drive so dataset files survive a disconnect. Without it
you re-download on every session.


In [ ]:
!pip install -q -r requirements.txt

import os
PROJECT_DIR = '/content/drive/MyDrive/medvlm'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['HF_HOME'] = f'{PROJECT_DIR}/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

FIG_DIR = f'{PROJECT_DIR}/figures'
print('figures ->', FIG_DIR)


## 3. Render figures inline

`explore.py` defaults to a non-interactive backend so the CLI works on a
headless machine. In a notebook we switch it back so plots appear here *and*
get written to Drive.


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

from pathlib import Path
from src import explore
print('ready')


## 4. Text statistics

Full split, text columns only — no images decoded, so this is fast even on
PathVQA.

**Write down the majority-class baseline.** It is the single most important
number on this page.


In [ ]:
stats = []
for name in ['vqa_rad', 'slake']:
    s = explore.text_stats(name)
    explore.print_stats(s)
    stats.append(s)


## 5. Composition and the floor to beat


In [ ]:
out = Path(FIG_DIR); out.mkdir(parents=True, exist_ok=True)
_ = explore.fig_composition(stats, out / 'composition.png')


## 6. Answer length — the case for the constrained prompt

Gold answers cluster at 1–2 words. A base model replying in 30 words of
clinical prose is not wrong about medicine; it is wrong about format. That
distinction is the spine of this project.


In [ ]:
_ = explore.fig_answer_lengths(stats, out / 'answer_lengths.png')


## 7. How concentrated is the answer space?

If a handful of answers cover most open-ended questions, then open-ended
recall is easier than it looks and you should say so in the report.


In [ ]:
for s in stats:
    _ = explore.fig_top_answers(s, out / f"{s['name']}_top_answers.png")


In [ ]:
for s in stats:
    _ = explore.fig_question_openers(s, out / f"{s['name']}_question_openers.png")


## 8. What the images actually look like

**This is the figure for your report.** It shows in one glance what three
paragraphs cannot: grayscale radiographs, far outside the vision encoder's
pretraining distribution, paired with short clinical questions.

Question titles are coloured by type — blue for closed, orange for open.


In [ ]:
_ = explore.fig_image_grid('vqa_rad', n=8, seed=0, out=out / 'vqa_rad_examples.png')


In [ ]:
_ = explore.fig_image_grid('slake', n=8, seed=0, out=out / 'slake_examples.png')


## 9. Image dimensions vs the token budget

Qwen-VL uses *dynamic* resolution: a larger image silently becomes more
tokens, and attention cost is quadratic in that. The dashed line is the
`max_pixels` cap set in `model.py`.

If most images sit above it, the cap is actively reshaping your inputs — and
since it must be held constant across every model you compare, that deserves
a sentence in the methods section.


In [ ]:
for name in ['vqa_rad', 'slake']:
    _ = explore.fig_image_sizes(name, n=250, seed=0, out=out / f'{name}_image_sizes.png')


## 10. Read raw examples

Numbers hide things. Read twenty real question–answer pairs and you will
notice conventions the statistics cannot show you — abbreviations, laterality,
how modality questions are phrased.


In [ ]:
from src.data import load_benchmark, iter_examples

ds, spec = load_benchmark('slake')
for i, ex in enumerate(iter_examples(ds.select(range(20)), spec)):
    print(f"[{ex['answer_type']:6s}] {ex['question']}")
    print(f"          -> {ex['answer']!r}\n")


## 11. Or just run everything from the CLI

Equivalent to sections 4–9. This is the command that goes in your README,
because it regenerates every figure from a clean checkout in one line.


In [ ]:
!python -m src.explore --dataset vqa_rad --dataset slake --out /content/drive/MyDrive/medvlm/figures


---

### Before you move on, write these down

- [ ] Majority-class baseline for **each** dataset — the real floor
- [ ] Closed/open split (SLAKE's is official; VQA-RAD's is inferred)
- [ ] Mean gold answer length
- [ ] Whether most images exceed the `max_pixels` cap

Then go to `01_zero_shot_baseline.ipynb`, which needs a **T4 GPU**.
